# WF2024 State Coverage Diagnostic V1

Purpose: follow `warm_start_action_cap_validation_v1`, where the selected sequence-risk action cap fixed `wf_2022` but failed `wf_2023_2024` with zero cap-trigger days. This notebook identifies the bad 2024 intervals, feature differences, and state-coverage gap before any new threshold or training run.

Governance:
- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `warm_start_action_cap_validation_v1`.
- **Role**: `diagnose`.
- **Target window**: `wf_2023_2024`; target is `+30pp` relative return and `<=15%` MaxDD.
- **Changed layer**: `model_selection`, `state_coverage_diagnostic`; no actor, reward, network, or runtime overlay change.
- **External reference**: Qlib workflow, probability of backtest overfitting, time-series momentum, momentum crashes, and volatility-managed portfolios: https://github.com/microsoft/qlib, https://arxiv.org/abs/2009.11189, https://doi.org/10.21314/JCF.2016.322, https://doi.org/10.1016/j.jfineco.2011.11.003, https://doi.org/10.1016/j.jfineco.2016.05.002, https://doi.org/10.1111/jofi.12513.
- **Hypothesis**: the failed 2024 fold is not another sequence-break state. It is a state coverage gap: low-exposure trend participation misses strong rally windows, while residual exposure still produces a later drawdown breach.
- **Acceptance criteria**: identify rolling poor windows, classify failure mode, quantify sequence-cap coverage, rank feature differences, and select the next mechanism without running another cap grid.
- **Failure mode**: if poor windows are drawdown-control only and sequence-risk state actually covers them, return to action-cap/checkpoint scoring; otherwise pivot to trend participation state plus drawdown guard.
- **Comparison-table row key**: `wf2024_state_coverage_diagnostic_v1`.

Historical duplicate check:
- Closest success case: `warm_start_action_cap_validation_v1` showed the cap improves `wf_2022` drawdown and relative return.
- Closest failure case: the same card showed no `wf_2023_2024` cap triggers and failed both relative and MaxDD gates.
- This run is not a duplicate because it analyzes the uncovered 2024 state and feature regime instead of rerunning the action-cap threshold.


In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.oos_diagnostics import (
    attribute_gate_gaps,
    classify_poor_window,
    compute_feature_differences,
    find_poor_windows,
    summarize_gate_gap_attribution,
)
from src.soxl_quant.app.path_diagnostics import find_worst_drawdown_intervals
from src.soxl_quant.data import build_dataset, init_qlib, segment_frame
from src.soxl_quant.settings import QLIB_DATA_PATH, load_profile

EXPERIMENT = 'wf2024_state_coverage_diagnostic_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'warm_start_action_cap_validation_v1'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
SELECTED = 'seq_cap10_two_of_three'
FOLD = 'wf_2023_2024'
TEST_SEG = ('2024-01-01', '2024-12-31')
TRAIN_SEG = ('2010-03-11', '2022-12-31')
VALID_SEG = ('2023-01-01', '2023-12-31')


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return str(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + chr(10), encoding='utf-8')

print('card', CARD)


card /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/wf2024_state_coverage_diagnostic_v1


In [2]:
source_selection = json.loads((SOURCE_CARD / 'selection.json').read_text(encoding='utf-8'))
source_metrics = pd.read_csv(SOURCE_CARD / 'warm_start_action_cap_metrics.csv')
source_trace = pd.read_csv(SOURCE_CARD / 'warm_start_decision_trace.csv')
thresholds = pd.read_csv(SOURCE_CARD / 'fold_sequence_thresholds.csv')
selected_metrics = source_metrics[source_metrics['candidate'].eq(SELECTED) & source_metrics['fold'].eq(FOLD)].iloc[0].to_dict()
baseline_metrics = source_metrics[source_metrics['candidate'].eq('baseline_no_action_cap') & source_metrics['fold'].eq(FOLD)].iloc[0].to_dict()

profile = load_profile(PROFILE_NAME)
init_qlib(QLIB_DATA_PATH)
dataset = build_dataset(
    train_seg=TRAIN_SEG,
    valid_seg=VALID_SEG,
    test_seg=TEST_SEG,
    feature_packs=tuple(profile.feature.packs),
    profile_name=profile.name,
)
features = segment_frame(dataset, 'test')
price = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()

daily = pd.read_csv(SOURCE_CARD / f'{SELECTED}_{FOLD}_daily_path.csv', parse_dates=['datetime'])
price_window = price.loc[daily['datetime'].min():daily['datetime'].max()].rename('price')
daily = daily.merge(price_window, left_on='datetime', right_index=True, how='left')
first_account = float(daily['account'].iloc[0])
first_price = float(daily['price'].iloc[0])
daily_oos = pd.DataFrame({
    'datetime': daily['datetime'],
    'strategy_nav': daily['account'].astype(float),
    'buyhold_nav': first_account * daily['price'].astype(float) / first_price,
    'position': daily['position'].astype(float),
})
daily_oos.to_csv(CARD / 'wf2024_selected_daily_oos.csv', index=False)

source_snapshot = pd.DataFrame([selected_metrics, baseline_metrics])
source_snapshot[['candidate', 'fold', 'return', 'buy_and_hold_return', 'relative_return', 'max_dd', 'avg_pos', 'trace_target_cap_changed_days']]


[27069:MainThread](2026-05-26 12:15:47,356) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[27069:MainThread](2026-05-26 12:15:48,126) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[27069:MainThread](2026-05-26 12:15:48,126) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[27069:MainThread](2026-05-26 12:15:50,381) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[27069:MainThread](2026-05-26 12:15:50,383) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[27069:MainThread](2026-05-26 12:15:50,384) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | Init data Done


,candidate,fold,return,buy_and_hold_return,relative_return,max_dd,avg_pos,trace_target_cap_changed_days
0,seq_cap10_two_of_three,wf_2023_2024,2.769492,-0.356635,3.126126,20.232446,0.160557,0
1,baseline_no_action_cap,wf_2023_2024,2.769492,-0.356635,3.126126,20.232446,0.160557,0


In [3]:
poor_21 = find_poor_windows(
    daily_oos,
    window_size=21,
    top_n=12,
    target_excess=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
)
poor_63 = find_poor_windows(
    daily_oos,
    window_size=63,
    top_n=12,
    target_excess=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
)
for frame_, label in ((poor_21, 'poor_windows_21'), (poor_63, 'poor_windows_63')):
    if not frame_.empty:
        frame_['failure_type'] = frame_.apply(
            lambda row: classify_poor_window(row, target_excess=TARGET_RELATIVE_RETURN, target_max_dd=TARGET_MAX_DD),
            axis=1,
        )
    frame_.to_csv(CARD / f'{label}.csv', index=False)

poor_21.head(8), poor_63.head(8)


(        start         end  rows  strategy_return  buyhold_return  \
 0  2024-02-07  2024-03-07    21        11.055851       56.447960   
 1  2024-04-22  2024-05-20    21         5.380465       50.452137   
 2  2024-04-19  2024-05-17    21         5.027549       47.450472   
 3  2024-09-06  2024-10-04    21         1.805570       41.294305   
 4  2024-05-01  2024-05-30    21         6.079282       45.475710   
 5  2024-02-06  2024-03-06    21        11.605916       48.448119   
 6  2024-02-02  2024-03-04    21        11.709061       47.904898   
 7  2024-02-01  2024-03-01    21        12.799390       48.936828   
 
    excess_return    max_dd   avg_pos  excess_gap  max_dd_gap  bad_score  \
 0     -45.392109  1.063455  0.150242   75.392109         0.0  75.392109   
 1     -45.071672  2.826736  0.138763   75.071672         0.0  75.071672   
 2     -42.422923  2.826736  0.136982   72.422923         0.0  72.422923   
 3     -39.488735  2.367649  0.088688   69.488735         0.0  69.488735 

In [4]:
feature_diffs_21 = compute_feature_differences(features, poor_21, daily=daily_oos, top_n=30)
feature_diffs_63 = compute_feature_differences(features, poor_63, daily=daily_oos, top_n=30)
feature_diffs_21.to_csv(CARD / 'feature_differences_21.csv', index=False)
feature_diffs_63.to_csv(CARD / 'feature_differences_63.csv', index=False)

feature_diffs_63[['feature', 'delta_z', 'bad_mean', 'reference_mean', 'spearman_to_excess']].head(15)


,feature,delta_z,bad_mean,reference_mean,spearman_to_excess
0,anchor_gap_180,4.939154,0.524131,-0.191953,-0.159637
1,mom_63,2.586750,0.468468,-0.069814,-0.073157
2,trend_slope_63,2.484985,0.140045,-0.063368,0.022172
3,vix_percentile_252,-2.411212,0.308100,0.763175,0.213634
4,trend_gap_63,2.199725,0.138776,-0.099698,0.022492
5,drawdown_63,1.913159,-0.135712,-0.375583,-0.174784
6,drawdown_streak_63,-1.843702,0.280998,0.789195,0.026432
7,days_since_high_63,-1.748458,0.262020,0.673768,0.014475
8,below_anchor_ratio_63,-1.742261,0.042098,0.657895,0.025531
9,anchor_slope_180,1.720400,0.094872,0.015071,0.018515


In [5]:
drawdowns = find_worst_drawdown_intervals(daily, top_n=8, min_depth=0.05)
drawdowns.to_csv(CARD / 'drawdown_intervals.csv', index=False)

selected_trace_2024 = source_trace[source_trace['candidate'].eq(SELECTED) & source_trace['fold'].eq(FOLD)].copy()
selected_trace_2024['date'] = pd.to_datetime(selected_trace_2024['date'])
threshold_row = thresholds[thresholds['fold'].eq(FOLD)].iloc[0]
state_frame = features.copy()
state_frame['sequence_anchor_slope_ready'] = state_frame['anchor_slope_180'].astype(float) <= float(threshold_row['sequence_risk_anchor_slope_max'])
state_frame['sequence_anchor_gap_ready'] = state_frame['anchor_gap_180'].astype(float) <= float(threshold_row['sequence_risk_anchor_gap_max'])
state_frame['sequence_trend_gap_ready'] = state_frame['trend_gap_63'].astype(float) <= float(threshold_row['sequence_risk_trend_gap_min'])
state_frame['sequence_hit_count'] = state_frame[['sequence_anchor_slope_ready', 'sequence_anchor_gap_ready', 'sequence_trend_gap_ready']].sum(axis=1)
state_frame['sequence_two_of_three'] = state_frame['sequence_hit_count'] >= 2
state_frame['high_trend_state'] = (
    (state_frame['anchor_gap_180'].astype(float) > state_frame['anchor_gap_180'].quantile(0.75))
    & (state_frame['mom_63'].astype(float) > state_frame['mom_63'].quantile(0.75))
)
state_frame['low_vix_trend_state'] = state_frame['high_trend_state'] & (state_frame['vix_percentile_252'].astype(float) < state_frame['vix_percentile_252'].quantile(0.40))

poor_63_dates = set()
for row in poor_63.itertuples(index=False):
    idx = pd.date_range(pd.Timestamp(row.start), pd.Timestamp(row.end), freq='D')
    poor_63_dates.update(pd.Timestamp(x).normalize() for x in idx)
feature_dates = pd.DatetimeIndex(pd.to_datetime(state_frame.index).normalize())
state_frame['in_top63_poor_window'] = feature_dates.isin(poor_63_dates)

coverage_rows = []
for label, mask in {
    'all_wf2024': pd.Series(True, index=state_frame.index),
    'top63_poor_windows': state_frame['in_top63_poor_window'],
    'high_trend_state': state_frame['high_trend_state'],
    'low_vix_trend_state': state_frame['low_vix_trend_state'],
}.items():
    sub = state_frame[mask]
    coverage_rows.append({
        'scope': label,
        'rows': int(len(sub)),
        'sequence_two_of_three_share': float(sub['sequence_two_of_three'].mean()) if len(sub) else 0.0,
        'mean_sequence_hit_count': float(sub['sequence_hit_count'].mean()) if len(sub) else 0.0,
        'mean_anchor_gap_180': float(sub['anchor_gap_180'].mean()) if len(sub) else float('nan'),
        'mean_mom_63': float(sub['mom_63'].mean()) if len(sub) else float('nan'),
        'mean_trend_gap_63': float(sub['trend_gap_63'].mean()) if len(sub) else float('nan'),
        'mean_vix_percentile_252': float(sub['vix_percentile_252'].mean()) if len(sub) else float('nan'),
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'state_coverage.csv', index=False)

trace_coverage = pd.DataFrame([{
    'fold': FOLD,
    'candidate': SELECTED,
    'trace_rows': int(len(selected_trace_2024)),
    'overlay_ready_days': int(selected_trace_2024['overlay_gate_ready'].fillna(False).astype(bool).sum()) if not selected_trace_2024.empty else 0,
    'target_cap_changed_days': int((pd.to_numeric(selected_trace_2024['target_after_overlay'], errors='coerce') < pd.to_numeric(selected_trace_2024['target_before_overlay'], errors='coerce') - 1e-9).sum()) if not selected_trace_2024.empty else 0,
    'first_trace_date': str(selected_trace_2024['date'].min().date()) if not selected_trace_2024.empty else '',
    'last_trace_date': str(selected_trace_2024['date'].max().date()) if not selected_trace_2024.empty else '',
}])
trace_coverage.to_csv(CARD / 'trace_coverage.csv', index=False)

coverage, trace_coverage, drawdowns.head(5)


(                 scope  rows  sequence_two_of_three_share  \
 0           all_wf2024   252                     0.107143   
 1   top63_poor_windows   138                     0.000000   
 2     high_trend_state    39                     0.000000   
 3  low_vix_trend_state    26                     0.000000   
 
    mean_sequence_hit_count  mean_anchor_gap_180  mean_mom_63  \
 0                 0.750000             0.200188     0.224960   
 1                 0.347826             0.524131     0.468468   
 2                 0.051282             0.707347     0.680064   
 3                 0.000000             0.694908     0.669183   
 
    mean_trend_gap_63  mean_vix_percentile_252  
 0           0.030895                 0.513968  
 1           0.138776                 0.308100  
 2           0.201532                 0.311762  
 3           0.204702                 0.214744  ,
            fold               candidate  trace_rows  overlay_ready_days  \
 0  wf_2023_2024  seq_cap10_two_of_thre

In [6]:
gate_metrics = {
    'selected_cap': {
        FOLD: {
            'return': selected_metrics['return'],
            'buy_and_hold_return': selected_metrics['buy_and_hold_return'],
            'relative_return': selected_metrics['relative_return'],
            'max_dd': selected_metrics['max_dd'],
            'avg_pos': selected_metrics['avg_pos'],
        }
    },
    'baseline_no_cap': {
        FOLD: {
            'return': baseline_metrics['return'],
            'buy_and_hold_return': baseline_metrics['buy_and_hold_return'],
            'relative_return': baseline_metrics['relative_return'],
            'max_dd': baseline_metrics['max_dd'],
            'avg_pos': baseline_metrics['avg_pos'],
        }
    },
}
gate_gaps = attribute_gate_gaps(gate_metrics, target_relative_return=TARGET_RELATIVE_RETURN, target_max_dd=TARGET_MAX_DD)
gate_summary = summarize_gate_gap_attribution(gate_gaps)
gate_gaps.to_csv(CARD / 'gate_gap_attribution.csv', index=False)
gate_summary.to_csv(CARD / 'gate_gap_profile_summary.csv', index=False)

failure_counts_21 = Counter(poor_21.get('failure_type', pd.Series(dtype=str)).dropna().astype(str))
failure_counts_63 = Counter(poor_63.get('failure_type', pd.Series(dtype=str)).dropna().astype(str))
top63_features = feature_diffs_63.head(8)['feature'].tolist() if not feature_diffs_63.empty else []
top21_features = feature_diffs_21.head(8)['feature'].tolist() if not feature_diffs_21.empty else []

selected_trace_changes = int(trace_coverage['target_cap_changed_days'].iloc[0])
selected_gate_pass = bool(float(selected_metrics['relative_return']) >= TARGET_RELATIVE_RETURN and float(selected_metrics['max_dd']) <= TARGET_MAX_DD)
rolling_underparticipation = failure_counts_63.get('underparticipation', 0) >= max(1, len(poor_63) // 2)
sequence_no_coverage = selected_trace_changes == 0 and float(coverage.loc[coverage['scope'].eq('top63_poor_windows'), 'sequence_two_of_three_share'].iloc[0]) < 0.05

if (not selected_gate_pass) and rolling_underparticipation and sequence_no_coverage:
    decision = 'wf2024_underparticipation_state_coverage_gap_no_promotion'
    next_mechanism = 'screen_trend_participation_state_with_drawdown_guard_before_training'
elif not selected_gate_pass:
    decision = 'wf2024_gate_failed_state_diagnostic_no_promotion'
    next_mechanism = 'aggregate_poor_windows_then_checkpoint_or_training_objective'
else:
    decision = 'wf2024_state_coverage_pass_diagnostic_only'
    next_mechanism = 'validate_fixed_split_and_wf2020_before_promotion'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'role': 'diagnose',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'warm_start_action_cap_validation_v1',
    'profile': PROFILE_NAME,
    'fold': FOLD,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'decision': decision,
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'selected_candidate': SELECTED,
    'selected_relative_return': float(selected_metrics['relative_return']),
    'selected_max_dd': float(selected_metrics['max_dd']),
    'selected_avg_pos': float(selected_metrics['avg_pos']),
    'selected_target_cap_changed_days': selected_trace_changes,
    'top21_failure_counts': dict(failure_counts_21),
    'top63_failure_counts': dict(failure_counts_63),
    'top63_features': top63_features,
    'top21_features': top21_features,
    'sequence_two_of_three_share_top63_poor_windows': float(coverage.loc[coverage['scope'].eq('top63_poor_windows'), 'sequence_two_of_three_share'].iloc[0]),
    'largest_drawdown_max_dd': float(drawdowns['max_dd'].iloc[0]) if not drawdowns.empty else None,
    'largest_drawdown_avg_pos': float(drawdowns['avg_pos'].iloc[0]) if not drawdowns.empty else None,
    'gate_failure_type': str(gate_gaps['failure_type'].iloc[0]) if not gate_gaps.empty else '',
    'next_mechanism': next_mechanism,
    'promotion_blocker': 'Diagnostic only: no new actor, no fixed-split/wf2020 validation, and 2024 fold still fails +30pp / <=15% gate.',
    'sidecars': [
        'wf2024_selected_daily_oos.csv', 'poor_windows_21.csv', 'poor_windows_63.csv',
        'feature_differences_21.csv', 'feature_differences_63.csv', 'drawdown_intervals.csv',
        'state_coverage.csv', 'trace_coverage.csv', 'gate_gap_attribution.csv',
        'gate_gap_profile_summary.csv', 'selection.json', 'summary.json',
    ],
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://doi.org/10.21314/JCF.2016.322',
        'https://doi.org/10.1016/j.jfineco.2011.11.003',
        'https://doi.org/10.1016/j.jfineco.2016.05.002',
        'https://doi.org/10.1111/jofi.12513',
    ],
}
summary = {
    **selection,
    'source_decision': source_selection.get('decision'),
    'selected_metrics': selected_metrics,
    'baseline_metrics': baseline_metrics,
    'poor_windows_21': poor_21.to_dict('records'),
    'poor_windows_63': poor_63.to_dict('records'),
    'feature_differences_21_top10': feature_diffs_21.head(10).to_dict('records'),
    'feature_differences_63_top10': feature_diffs_63.head(10).to_dict('records'),
    'state_coverage': coverage.to_dict('records'),
    'trace_coverage': trace_coverage.to_dict('records'),
    'drawdown_intervals': drawdowns.to_dict('records'),
    'gate_gap_attribution': gate_gaps.to_dict('records'),
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
selection


{'experiment': 'wf2024_state_coverage_diagnostic_v1',
 'created_at': '2026-05-26T04:15:51.007690+00:00',
 'role': 'diagnose',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'warm_start_action_cap_validation_v1',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'fold': 'wf_2023_2024',
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 15.0,
 'decision': 'wf2024_underparticipation_state_coverage_gap_no_promotion',
 'promotion_allowed': False,
 'training_allowed_from_this_card': False,
 'selected_candidate': 'seq_cap10_two_of_three',
 'selected_relative_return': 3.126126377580462,
 'selected_max_dd': 20.23244637943855,
 'selected_avg_pos': 0.1605570264071269,
 'selected_target_cap_changed_days': 0,
 'top21_failure_counts': {'underparticipation': 12},
 'top63_failure_counts': {'underparticipation': 12},
 'top63_features': ['anchor_gap_180',
  'mom_63',
  'trend_slope_63',
  'vix_percentile_252',
  'trend_gap_63',
  'drawdown_63',
  'dra

In [7]:
top63 = feature_diffs_63[['feature', 'delta_z', 'bad_mean', 'reference_mean']].head(8)
top21 = feature_diffs_21[['feature', 'delta_z', 'bad_mean', 'reference_mean']].head(8)
poor63_first = poor_63.head(3)[['start', 'end', 'excess_return', 'max_dd', 'avg_pos', 'failure_type']]
dd_first = drawdowns.head(3)[['start', 'end', 'max_dd', 'avg_pos', 'return']]

conclusion = f"""
## Conclusion

Decision: `{selection['decision']}`. Promotion remains blocked.

2024 warm-start selected path still fails the target: relative `{selection['selected_relative_return']:.2f}pp`, MaxDD `{selection['selected_max_dd']:.2f}%`, avg position `{selection['selected_avg_pos']:.3f}`. The sequence action cap changed target on `{selection['selected_target_cap_changed_days']}` days in this fold, and the top 63-day poor-window sequence two-of-three coverage share is `{selection['sequence_two_of_three_share_top63_poor_windows']:.3f}`.

Rolling poor-window diagnosis: 21-day failure counts `{selection['top21_failure_counts']}`; 63-day failure counts `{selection['top63_failure_counts']}`. The worst 63-day windows are underparticipation windows, not sequence-breakdown windows.

Top 63-day feature deltas: `{', '.join(selection['top63_features'][:8])}`. These point to a high-trend / low-volatility participation gap: high `anchor_gap_180`, `mom_63`, and trend-slope features with low VIX percentile. The largest drawdown interval still reaches `{selection['largest_drawdown_max_dd']:.2f}%` with avg position `{selection['largest_drawdown_avg_pos']:.3f}`, so the next attempt needs trend participation plus drawdown guard, not a pure long-only participation lift.

Next action: `{selection['next_mechanism']}`. Do not run another action-cap threshold grid from this evidence; screen a trend-participation state conditioned by drawdown/volatility guard, or route this into checkpoint/training-objective design.

Artifacts: `poor_windows_21.csv`, `poor_windows_63.csv`, `feature_differences_21.csv`, `feature_differences_63.csv`, `drawdown_intervals.csv`, `state_coverage.csv`, `trace_coverage.csv`, `gate_gap_attribution.csv`, `selection.json`, `summary.json`.
""".strip()

display(Markdown(conclusion))
display(Markdown('### Worst 63-day windows'))
display(poor63_first)
display(Markdown('### Top 63-day feature deltas'))
display(top63)
display(Markdown('### Drawdown intervals'))
display(dd_first)


## Conclusion

Decision: `wf2024_underparticipation_state_coverage_gap_no_promotion`. Promotion remains blocked.

2024 warm-start selected path still fails the target: relative `3.13pp`, MaxDD `20.23%`, avg position `0.161`. The sequence action cap changed target on `0` days in this fold, and the top 63-day poor-window sequence two-of-three coverage share is `0.000`.

Rolling poor-window diagnosis: 21-day failure counts `{'underparticipation': 12}`; 63-day failure counts `{'underparticipation': 12}`. The worst 63-day windows are underparticipation windows, not sequence-breakdown windows.

Top 63-day feature deltas: `anchor_gap_180, mom_63, trend_slope_63, vix_percentile_252, trend_gap_63, drawdown_63, drawdown_streak_63, days_since_high_63`. These point to a high-trend / low-volatility participation gap: high `anchor_gap_180`, `mom_63`, and trend-slope features with low VIX percentile. The largest drawdown interval still reaches `20.23%` with avg position `0.130`, so the next attempt needs trend participation plus drawdown guard, not a pure long-only participation lift.

Next action: `screen_trend_participation_state_with_drawdown_guard_before_training`. Do not run another action-cap threshold grid from this evidence; screen a trend-participation state conditioned by drawdown/volatility guard, or route this into checkpoint/training-objective design.

Artifacts: `poor_windows_21.csv`, `poor_windows_63.csv`, `feature_differences_21.csv`, `feature_differences_63.csv`, `drawdown_intervals.csv`, `state_coverage.csv`, `trace_coverage.csv`, `gate_gap_attribution.csv`, `selection.json`, `summary.json`.

### Worst 63-day windows

,start,end,excess_return,max_dd,avg_pos,failure_type
0,2024-04-22,2024-07-22,-64.046050,12.867847,0.218922,underparticipation
1,2024-04-16,2024-07-16,-59.862732,8.008688,0.217077,underparticipation
2,2024-01-03,2024-04-03,-59.312532,3.874673,0.172988,underparticipation


### Top 63-day feature deltas

,feature,delta_z,bad_mean,reference_mean
0,anchor_gap_180,4.939154,0.524131,-0.191953
1,mom_63,2.586750,0.468468,-0.069814
2,trend_slope_63,2.484985,0.140045,-0.063368
3,vix_percentile_252,-2.411212,0.308100,0.763175
4,trend_gap_63,2.199725,0.138776,-0.099698
5,drawdown_63,1.913159,-0.135712,-0.375583
6,drawdown_streak_63,-1.843702,0.280998,0.789195
7,days_since_high_63,-1.748458,0.262020,0.673768


### Drawdown intervals

,start,end,max_dd,avg_pos,return
0,2024-07-10,2024-12-19,20.2324,0.129568,-20.2324
1,2024-06-18,2024-06-24,8.0087,0.414159,-8.0087
2,2024-03-07,2024-05-01,7.6074,0.149270,-7.6074
